In [11]:
import os
import torch
import torch.nn as nn
from torchvision.transforms import v2 as transforms
import librosa
import numpy as np
from sklearn.metrics import roc_auc_score, mean_squared_error
from Audiopy_ML import autoaudio
import glob

In [12]:
generator = torch.Generator().manual_seed(42)
np.random.seed(42)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [14]:
class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir, train):
        self.audio_dir = audio_dir
        file_list = os.listdir(audio_dir)

        labels = np.zeros(len(file_list), dtype=int) if train else [1 if el[0] == 'a' else 0 for el in file_list]
        self.labels = torch.tensor(labels, dtype=torch.int8).to(device)

        loads = [librosa.load(os.path.join(audio_dir, el), sr=None) for el in file_list]
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.spect_dbs[idx], self.labels[idx]

In [15]:
train_dataset = glob.glob('archive/dev_data/dev_data/slider/train/*')
test_dataset = glob.glob('archive/dev_data/dev_data/slider/test/*')

In [16]:
df_train =autoaudio.AutomatedExtractor_multiple(train_dataset)
df_test =autoaudio.AutomatedExtractor_multiple(test_dataset)

df_train = df_train.applymap(lambda x: np.median(x))
df_test = df_test.applymap(lambda x: np.median(x))


KeyboardInterrupt: 

In [ ]:
x=np.array(df_train).tolist()
x=np.array(x)
x_t=np.array(df_test).tolist()
x_t=np.array(x_t)

In [ ]:
from sklearn.preprocessing import StandardScaler, Normalizer
x=Normalizer().fit_transform(x)
x_t=Normalizer().fit_transform(x_t)
x=StandardScaler().fit_transform(x)
x_t=StandardScaler().fit_transform(x_t)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(7, 64),
            nn.ELU(),
            nn.Linear(64, 32),
            nn.ELU(),
            nn.Linear(32, 16),
            nn.ELU(),
            nn.Linear(16, 8),
            nn.ELU(),
            nn.Linear(8, 4),
            nn.ELU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(4, 8),
            nn.ELU(),
            nn.Linear(8, 16),
            nn.ELU(),
            nn.Linear(16, 32),
            nn.ELU(),
            nn.Linear(32, 64),
            nn.ELU(),
            nn.Linear(64, 7),
            nn.ELU()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Assume x and x_t are NumPy arrays with shape [num_samples, 7]
x_tensor = torch.tensor(x, dtype=torch.float32)
x_t_tensor = torch.tensor(x_t, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(x_tensor, x_tensor), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x_t_tensor, x_t_tensor), batch_size=32, shuffle=False)

# Model, optimizer, loss
model = Autoencoder()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.05, patience=2, verbose=True)


In [ ]:
num_epochs = 150

for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for batch_x, _ in train_loader:
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_x)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch_x, _ in val_loader:
            output = model(batch_x)
            loss = criterion(output, batch_x)
            val_losses.append(loss.item())
            all_preds.append(output.numpy())
            all_targets.append(batch_x.numpy())

    avg_train_loss = np.mean(train_losses)
    avg_val_loss = np.mean(val_losses)
    
    # Optional metrics
    preds = np.vstack(all_preds)
    targets = np.vstack(all_targets)
    mae = mean_absolute_error(targets, preds)
    try:
        msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
    except:
        msle = float("nan")

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f} - MAE: {mae:.4f} - MSLE: {msle:.4f}")

    # Adjust learning rate
    scheduler.step(avg_val_loss)